# 02_8. Steps 8–9 with other models / Шаги 8–9 на других моделях

**EN.** Steps 8 (model importance) and 9 (group ablation) of the selection for Y1 (population: loans current at t, DPD = 0, never in default before t; 59 candidates, 18 selected), and the non-inferiority comparison, were run with CatBoost, the main model of the thesis. A model-based check can favour the features that suit that particular model. Here the same three checks are repeated with four other learners of different kinds, with the same train, validation and rules:

- **logistic regression on WoE** — a linear model on the train WoE values of every feature, the banking scorecard standard (L2, C = 1);
- **XGBoost** — gradient boosting with another tree algorithm; categoricals passed natively;
- **LightGBM** — gradient boosting with leaf-wise trees;

  both with the Y1 settings found by the Optuna tuning of stage 3 (XGBoost: 368 trees, depth 9, learning rate 0.036, L2 3.1, minimum leaf weight 402, row subsample 0.85; LightGBM: 335 trees, depth 7, 128 leaves, learning rate 0.023, L2 14.6, minimum leaf 1,581 rows, row subsample 0.82). A first run with untuned settings (300 trees, learning rate 0.1, no leaf minimum) overfitted the rare event — validation AUC 0.89 (XGBoost) and 0.79 (LightGBM), with negative permutation importance for the strongest features — so its results were discarded;
- **random forest** — averaged independent deep trees (300 trees, leaves of at least 50 rows); categoricals enter as their train WoE values; for memory and time it is trained on a 400,000-row train subsample (all events kept, weights rescaled).

The selection is **not** changed by this notebook: it is a robustness check. If another learner disagrees with a CatBoost decision, the disagreement is reported and discussed.

**RU.** Шаги 8 (важность в модели) и 9 (абляция по группам) отбора для Y1 (популяция — кредиты без просрочки на t, DPD = 0, не бывшие в дефолте до t; 59 кандидатов, 18 отобранных), а также сравнение «не хуже», выполнялись на CatBoost — основной модели диссертации. Проверка на модели может предпочесть признаки, удобные именно этой модели. Здесь те же три проверки повторяются на четырёх других алгоритмах разного типа, с тем же train, validation и правилами:

- **логистическая регрессия на WoE** — линейная модель на WoE-значениях train всех признаков, банковский стандарт скоринговой карты (L2, C = 1);
- **XGBoost** — градиентный бустинг с другим алгоритмом построения деревьев; категориальные признаки подаются как категории;
- **LightGBM** — градиентный бустинг с ростом деревьев по листьям;

  оба с настройками для Y1, найденными подбором Optuna на этапе 3 (XGBoost: 368 деревьев, глубина 9, шаг 0,036, L2 3,1, минимальный вес листа 402, доля строк 0,85; LightGBM: 335 деревьев, глубина 7, 128 листьев, шаг 0,023, L2 14,6, в листе не меньше 1 581 строки, доля строк 0,82). Первый прогон с настройками без подбора (300 деревьев, шаг 0,1, без минимума листа) переобучился на редком событии — AUC на validation 0,89 (XGBoost) и 0,79 (LightGBM), с отрицательной permutation importance у сильнейших признаков, — поэтому его результаты отброшены;
- **случайный лес** — усреднение независимых глубоких деревьев (300 деревьев, в листе не меньше 50 строк); категориальные признаки подаются своими WoE-значениями train; из-за памяти и времени обучается на подвыборке train из 400 тыс. строк (все события сохранены, веса пересчитаны).

Отбор этот ноутбук **не меняет**: это проверка устойчивости. Если другой алгоритм не согласен с решением CatBoost, расхождение показывается и разбирается.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import gc
import numpy as np
import pandas as pd
import polars as pl
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from xai_ews.config import feature_config
from xai_ews.data.features import COLUMNS, candidate_features, categorical_features
from xai_ews.features.data import TASKS, CategoryEncoder, load_train, load_validation, subsample
from xai_ews.features.evaluation import compare_models
from xai_ews.features.other_models import (KINDS, boruta, fit_predict_validation, model_frame, native_importance,
                                            permutation_importance)
from xai_ews.features.univariate import weighted_auc
from xai_ews.paths import FEATURE_SELECTION_DIR, STAGE2_INTERIM_DIR

CFG = feature_config()
IMP_RULES, ABL_RULES = CFG["rules"]["importance"], CFG["rules"]["ablation"]
P = CFG["rules"]["other_models"]
SEED = CFG["data"]["seed"]
ROWS = {**{k: None for k in KINDS}, "random_forest": P["random_forest_train_rows"]}
ALL = candidate_features()
CATS = set(categorical_features())
red = pd.read_csv(FEATURE_SELECTION_DIR / "redundancy.csv")
imp_cb = pd.read_csv(FEATURE_SELECTION_DIR / "importance.csv")
CAND = red.loc[red["pass_redundancy"], "feature"].tolist()        # input of step 8
AFTER8 = imp_cb.loc[imp_cb["pass_importance"], "feature"].tolist()
FINAL = CFG["selected"]["features"]
print(f"tasks {list(TASKS)}; candidates {len(CAND)}, after step 8 (CatBoost) {len(AFTER8)}, selected {len(FINAL)}")

tasks ['y1']; candidates 21, after step 8 (CatBoost) 21, selected 18


### 1. Models on the candidates and on all 59 / Модели на кандидатах и на всех 59

**EN.** Each learner is fitted on the 21 candidates of step 8 (CatBoost removed nothing at step 8; the selected 18 = these minus the geography group removed at step 9, compared in section 4) and on all 59 features; AUC on validation without the hurricane rows (and with them). CatBoost values from 02_5 are added for comparison.

**RU.** Каждый алгоритм обучается на 21 кандидате шага 8 (CatBoost на шаге 8 ничего не удалил; отобранные 18 = эти признаки без группы «география», удалённой на шаге 9, они сравниваются в разделе 4) и на всех 59 признаках; AUC на validation без наблюдений урагана (и с ними). Для сравнения добавлены значения CatBoost из 02_5.

In [2]:
preds, models, aucs = {}, {}, []
for task in TASKS:
    for kind in KINDS:
        for name, feats in [("candidates", CAND), ("all", ALL)]:
            p, m, enc, binners = fit_predict_validation(kind, task, feats, P, f"{kind}_{name}", seed=SEED,
                                                        train_rows=ROWS[kind])
            preds[(task, kind, name)], models[(task, kind, name)] = p, (m, enc, binners, feats)
            main = p.filter(~pl.col("val_hurricane_2017"))
            aucs.append({"task": task, "model": kind, "features": name, "n": len(feats),
                         "auc_val": weighted_auc(main[task].to_numpy(), main["logit"].to_numpy()),
                         "auc_val_with_hurricanes": weighted_auc(p[task].to_numpy(), p["logit"].to_numpy())})
            gc.collect()
cb = (pd.read_csv(FEATURE_SELECTION_DIR / "importance_models_auc.csv")      # CatBoost, 02_5
      .rename(columns={"features": "n", "model": "features"}).assign(model="catboost"))
aucs = pd.concat([pd.DataFrame(aucs), cb[["task", "model", "features", "n", "auc_val", "auc_val_with_hurricanes"]]],
                 ignore_index=True)
aucs.pivot(index=["task", "features"], columns="model", values="auc_val")

model            catboost  lightgbm  logistic_woe  random_forest  xgboost
task features                                                            
y1   all           0.8509    0.8326        0.8471         0.8336   0.8561
     candidates    0.8510    0.8339        0.8420         0.8393   0.8550

**EN. How to read.** On the 21 candidates the five learners reach AUC 0.834–0.855: XGBoost 0.855, CatBoost 0.851, logistic regression 0.842, random forest 0.839, LightGBM 0.834. On all 59 features the range is 0.833–0.856: XGBoost 0.856, CatBoost 0.851, logistic regression 0.847, random forest 0.834, LightGBM 0.833 (the forest is clearly better on 21 than on 59). The AUCs are lower than in the earlier version of this stage (0.93) because the population is now loans current at t that have never been in default and the delinquency history is not used: the models have to warn before any payment is missed. LightGBM is the weakest learner on the 21 features; section 3 shows why (geography). All four learners are reasonable models for the checks below.

**RU. Как читать.** На 21 кандидате пять алгоритмов дают AUC 0,834–0,855: XGBoost 0,855, CatBoost 0,851, логистическая регрессия 0,842, случайный лес 0,839, LightGBM 0,834. На всех 59 признаках разброс 0,833–0,856: XGBoost 0,856, CatBoost 0,851, логистическая регрессия 0,847, случайный лес 0,834, LightGBM 0,833 (лесу 21 признак заметно лучше 59). AUC ниже, чем в прежней версии этого этапа (0,93), потому что популяция теперь — кредиты без просрочки на t, ни разу не бывшие в дефолте, а история просрочек не используется: модели должны предупредить раньше, чем пропущен хоть один платёж. Слабее всех на 21 признаке LightGBM; почему — видно в разделе 3 (география). Все четыре алгоритма — достаточно хорошие модели для проверок ниже.

### 2. Step 8: importance with each learner / Шаг 8: важность в каждом алгоритме

**EN.** Same rule as for CatBoost: a feature would be dropped only if **Boruta rejects it and its permutation importance is ≤ 0**. Permutation importance on 500,000 validation rows (3 repeats); Boruta on 400,000 train rows (20 rounds; importance: total gain for XGBoost and LightGBM, impurity decrease for the forest, |coefficient| × SD of the input for the logistic regression).

**RU.** То же правило, что для CatBoost: признак удалялся бы, только если **Boruta его отвергает и permutation importance ≤ 0**. Permutation importance — на 500 тыс. строк validation (3 повтора); Boruta — на 400 тыс. строк train (20 раундов; важность: суммарный gain у XGBoost и LightGBM, уменьшение неоднородности у леса, |коэффициент| × SD входа у логистической регрессии).

In [3]:
step8 = []
for task in TASKS:
    va = subsample(load_validation(task, CAND), task, IMP_RULES["validation_rows"], SEED)
    tr = subsample(load_train(task, CAND), task, IMP_RULES["train_rows"], SEED)
    for kind in KINDS:
        f_cache = STAGE2_INTERIM_DIR / f"step8_{kind}_{task}.csv"
        if f_cache.exists() and set(pd.read_csv(f_cache)["feature"]) == set(CAND):
            step8.append(pd.read_csv(f_cache)); continue
        m, enc, binners, feats = models[(task, kind, "candidates")]
        perm = permutation_importance(m, model_frame(va, CAND, enc, kind, binners), va[task].to_numpy(),
                                      va["sample_weight"].to_numpy(), repeats=IMP_RULES["permutation_repeats"], seed=SEED)
        enc_t = CategoryEncoder(CFG["rules"]["stability"]["high_cardinality_top_n"]).fit(tr, [c for c in CATS if c in CAND])
        bor = boruta(kind, model_frame(tr, CAND, enc_t, kind, binners), tr[task].to_numpy(), tr["sample_weight"].to_numpy(),
                     P, n_iter=IMP_RULES["boruta_iterations"], alpha=IMP_RULES["boruta_alpha"], seed=SEED)
        r = perm.merge(bor, on="feature").assign(model=kind, task=task)
        r["drop_rule"] = (r["boruta_decision"] == "rejected") & (r["perm_auc_drop"] <= 0)
        r.to_csv(f_cache, index=False)
        step8.append(r)
        gc.collect()
    del va, tr; gc.collect()
step8 = pd.concat(step8, ignore_index=True)
cb8 = imp_cb[["feature", "pass_importance", *[c for c in imp_cb.columns if c.startswith(("perm_auc_drop_y", "boruta_decision_y"))]]]
wide8 = step8.pivot_table(index="feature", columns="model", values=["perm_auc_drop", "boruta_decision", "drop_rule"], aggfunc="first")
wide8.columns = [f"{a}_{b}" for a, b in wide8.columns]
wide8 = wide8.join(cb8.set_index("feature").rename(columns=lambda c: c + "_catboost"))
print("dropped by the step-8 rule:", {k: step8.loc[step8["model"] == k, "feature"][step8.loc[step8["model"] == k, "drop_rule"]].tolist() for k in KINDS},
      "| CatBoost:", imp_cb.loc[~imp_cb["pass_importance"], "feature"].tolist())
wide8.sort_values("perm_auc_drop_xgboost", ascending=False)

dropped by the step-8 rule: {'logistic_woe': ['first_time_home_buyer_indicator', 'property_state'], 'xgboost': ['first_time_home_buyer_indicator'], 'lightgbm': ['property_type'], 'random_forest': ['property_type', 'seller_name', 'property_state']} | CatBoost: []


,boruta_decision_lightgbm,boruta_decision_logistic_woe,boruta_decision_random_forest,boruta_decision_xgboost,drop_rule_lightgbm,drop_rule_logistic_woe,drop_rule_random_forest,drop_rule_xgboost,perm_auc_drop_lightgbm,perm_auc_drop_logistic_woe,perm_auc_drop_random_forest,perm_auc_drop_xgboost,pass_importance_catboost,perm_auc_drop_y1_catboost,boruta_decision_y1_catboost
feature,,,,,,,,,,,,,,,
borrower_credit_score_at_origination,confirmed,confirmed,confirmed,confirmed,False,False,False,False,0.1016,0.1057,0.0971,0.0824,True,0.0971,confirmed
co_borrower_credit_score_at_origination,confirmed,confirmed,rejected,confirmed,False,False,False,False,0.0673,0.0344,0.0366,0.0478,True,0.0375,confirmed
excess_principal_12m,rejected,confirmed,rejected,confirmed,False,False,False,False,0.0071,0.0005,0.0070,0.0125,True,0.0068,confirmed
upb_vs_schedule,rejected,confirmed,rejected,confirmed,False,False,False,False,0.0067,0.0011,0.0092,0.0109,True,0.0115,confirmed
mtm_ltv,rejected,confirmed,confirmed,confirmed,False,False,False,False,0.0021,0.0014,0.0104,0.0078,True,0.0057,confirmed
debt_to_income_dti,rejected,confirmed,rejected,confirmed,False,False,False,False,0.0062,0.0061,0.0083,0.0069,True,0.0066,confirmed
original_interest_rate,rejected,confirmed,rejected,confirmed,False,False,False,False,0.0086,0.0064,0.0066,0.0068,True,0.0055,confirmed
n_curtail_6m,rejected,confirmed,rejected,confirmed,False,False,False,False,0.0087,0.0081,0.0018,0.0066,True,0.0051,confirmed
original_upb,rejected,confirmed,tentative,confirmed,False,False,False,False,0.0012,0.0007,0.0030,0.0054,True,0.0028,confirmed


**EN. How to read and conclusion.** CatBoost removes no feature at step 8. The other learners would remove 1–3 features each (XGBoost 1, LightGBM 1, logistic regression 2, random forest 3):
- `first_time_home_buyer_indicator` — logistic regression and XGBoost (permutation importance about 0 for both; with CatBoost −0.0001, one of its weakest features);
- `property_state` — logistic regression and the random forest (permutation importance about 0; with CatBoost +0.0001). It belongs to the geography group, which is removed at step 9 anyway;
- `property_type` — LightGBM and the random forest (about 0; with CatBoost +0.0003);
- `seller_name` — the random forest (−0.0002; with CatBoost also −0.0002, kept there because Boruta confirms it).

Boruta rejects many features with LightGBM and the random forest (LightGBM confirms only 3 of the 21 features, the forest 4) because their native importance is concentrated on a few strong features (FICO; for LightGBM also ZIP3) rather than on the shadows; the second condition of the rule (permutation ≤ 0 on validation) is what keeps them. The CatBoost decision of step 8 (keep all 21) is therefore largely shared. The weak cases — `first_time_home_buyer_indicator`, `property_type`, `seller_name` — add little for any learner and are kept by the rule; they are sensitivity checks for stage 3.

**RU. Как читать и вывод.** CatBoost на шаге 8 не удаляет ни одного признака. Другие алгоритмы удалили бы по 1–3 признака (XGBoost — 1, LightGBM — 1, логистическая регрессия — 2, случайный лес — 3):
- `first_time_home_buyer_indicator` — логистическая регрессия и XGBoost (permutation importance около 0 у обоих; у CatBoost −0,0001, один из самых слабых признаков);
- `property_state` — логистическая регрессия и случайный лес (permutation importance около 0; у CatBoost +0,0001). Он входит в группу «география», которая и так удаляется на шаге 9;
- `property_type` — LightGBM и случайный лес (около 0; у CatBoost +0,0003);
- `seller_name` — случайный лес (−0,0002; у CatBoost тоже −0,0002, там он остаётся, потому что Boruta его подтверждает).

Boruta у LightGBM и случайного леса отвергает много признаков (LightGBM подтверждает лишь 3 из 21, лес — 4), потому что их встроенная важность сосредоточена на нескольких сильных признаках (FICO, у LightGBM ещё ZIP3), а не на их тенях; признаки удерживает второе условие правила (permutation ≤ 0 на validation). Решение CatBoost на шаге 8 (оставить все 21) поэтому в основном разделяют и другие алгоритмы. Слабые случаи — `first_time_home_buyer_indicator`, `property_type`, `seller_name` — мало что добавляют любому алгоритму и остаются по правилу; на этапе 3 они переходят в проверки чувствительности.

### 3. Step 9: group ablation with each learner / Шаг 9: абляция по группам в каждом алгоритме

**EN.** Reference = the features after step 8. For each group a model without it; the group would be removed if the model without it is significantly better (lower bound of the 95 % cluster-bootstrap CI of ΔAUC > 0 and DeLong p < 0.05), exactly as in 02_6.

**RU.** Эталон — признаки после шага 8. Для каждой группы — модель без неё; группа удалялась бы, если модель без неё значимо лучше (нижняя граница 95 % кластерного bootstrap-интервала ΔAUC > 0 и DeLong p < 0,05), ровно как в 02_6.

In [4]:
GROUPS8 = sorted({COLUMNS[f].group for f in AFTER8})
sets8 = {"after_step8": AFTER8, **{f"minus_{g}": [f for f in AFTER8 if COLUMNS[f].group != g] for g in GROUPS8}}
step9 = []
for task in TASKS:
    for kind in KINDS:
        f_cache = STAGE2_INTERIM_DIR / f"step9_{kind}_{task}.csv"
        if f_cache.exists() and sorted(pd.read_csv(f_cache)["model"]) == sorted(sets8):
            step9.append(pd.read_csv(f_cache)); continue
        p9 = {n: (preds[(task, kind, "candidates")] if f == CAND else
                  fit_predict_validation(kind, task, f, P, f"{kind}_{n}", seed=SEED, train_rows=ROWS[kind])[0])
              for n, f in sets8.items()}
        r = compare_models(p9, task, "after_step8", b=ABL_RULES["bootstrap_b"], seed=SEED).assign(learner=kind, task=task)
        r["features"] = [len(sets8[n]) for n in r["model"]]
        r.to_csv(f_cache, index=False)
        step9.append(r)
        del p9; gc.collect()
step9 = pd.concat(step9, ignore_index=True)
step9["would_drop"] = (step9["model"] != "after_step8") & (step9["delta_lo"] > 0) & (step9["delong_p"] < 0.05)
cb9 = pd.concat([pd.read_csv(STAGE2_INTERIM_DIR / f"ablation_step9_{t}.csv").assign(learner="catboost", task=t) for t in TASKS])
cb9["would_drop"] = (cb9["model"] != "after_step8") & (cb9["delta_lo"] > 0) & (cb9["delong_p"] < 0.05)
both9 = pd.concat([step9, cb9], ignore_index=True)
print("groups that would be removed:", both9[both9["would_drop"]].groupby("learner")["model"].apply(list).to_dict())
both9.pivot_table(index=["task", "model"], columns="learner", values=["delta_auc", "delta_lo", "delta_hi", "delong_p"]).round(4)

groups that would be removed: {'catboost': ['minus_geography'], 'lightgbm': ['minus_geography']}


delong_p                                             delta_auc                                             delta_hi                                             delta_lo  \
learner               catboost lightgbm logistic_woe random_forest xgboost  catboost lightgbm logistic_woe random_forest xgboost catboost lightgbm logistic_woe random_forest xgboost catboost   
task model                                                                                                                                                                                       
y1   minus_behaviour    0.0000   0.0000       0.0000        0.0000  0.0000   -0.0123  -0.0124      -0.0075       -0.0099 -0.0123  -0.0094  -0.0098      -0.0059       -0.0069 -0.0099  -0.0152   
     minus_borrower     0.0000   0.0000       0.0000        0.0000  0.0000   -0.0778  -0.0815      -0.0911       -0.0951 -0.0692  -0.0697  -0.0734      -0.0827       -0.0862 -0.0619  -0.0854   
     minus_exposure     0.3374   0.0060       0.0605        0.2057  0.0107   -0.0013  -0.0022      -0.0005       -0.0021 -0.0025   0.0005  -0.0010      -0.0000        0.0003 -0.0008  -0.0032   
     minus_geography    0.0000   0.0000       0.2099        0.6773  0.1568    0.0036   0.0208      -0.0003       -0.0008 -0.0003   0.0055   0.0252       0.0016        0.0019  0.0009   0.0015   
     minus_loan         0.4876   0.0003       0.0000        0.0087  0.0000   -0.0022  -0.0021      -0.0032       -0.0036 -0.0043   0.0003  -0.0001      -0.0020       -0.0007 -0.0023  -0.0047   
     minus_macro        0.4158   0.0067       0.0177        0.1792  0.5241   -0.0013  -0.0021      -0.0005        0.0026 -0.0011   0.0007  -0.0009       0.0001        0.0052  0.0003  -0.0032   
     minus_state_at_t   0.4926   0.3936       0.0005        0.2636  0.4778    0.0002  -0.0005      -0.0002        0.0015 -0.0003   0.0019   0.0006      -0.0001        0.0039  0.0005  -0.0014   

                                                                   
learner               lightgbm logistic_woe random_forest xgboost  
task model                                                         
y1   minus_behaviour   -0.0149      -0.0092       -0.0132 -0.0148  
     minus_borrower    -0.0903      -0.0993       -0.1041 -0.0758  
     minus_exposure    -0.0034      -0.0009       -0.0047 -0.0040  
     minus_geography    0.0166      -0.0020       -0.0035 -0.0015  
     minus_loan        -0.0041      -0.0044       -0.0064 -0.0062  
     minus_macro       -0.0033      -0.0010        0.0001 -0.0024  
     minus_state_at_t  -0.0017      -0.0003       -0.0012 -0.0011

**EN. How to read and conclusion.**
- **Borrower and behaviour are essential for every learner**: without the borrower group AUC falls by 0.069–0.095, without behaviour by 0.008–0.012 (all DeLong p < 0.001). Behaviour here is the payment pattern of a loan that is still current (pace against the schedule, partial prepayments, balance changes), not the delinquency history.
- **Geography hurts two learners and is neutral for three.** Without state, MSA and ZIP3 LightGBM is much better (+0.0208 [0.0166; 0.0252], p < 0.001; AUC 0.855 instead of 0.834): it overfits the high-cardinality ZIP3, which takes 39 % of its gain in the model on all 59. CatBoost is also better without it (+0.0036 [0.0015; 0.0055], p < 0.001), so the rule removes the group (02_6). For the logistic regression (−0.0003), XGBoost (−0.0003) and the random forest (−0.0008) the CI includes 0: geography neither helps nor hurts them.
- **Exposure, loan and macro** are useful or neutral for every learner: exposure −0.0005 (logistic regression, p = 0.06) to −0.0025 (XGBoost, p = 0.01), loan −0.0021 (LightGBM) to −0.0043 (XGBoost, p < 0.001), macro −0.0021 (LightGBM, p = 0.007) to +0.0026 (forest, p = 0.18). **State at t** (`age`) is neutral (−0.0005…+0.0015); only the logistic regression gains a little from it (−0.0002 without it, p < 0.001).

By the rule fixed in advance the selection uses CatBoost; the removal of geography is shared by LightGBM, and no learner is significantly worse without it, so the 18 features stay. For stage 3 the sensitivity checks are the model **with geography** and the weakest features of step 8.

**RU. Как читать и вывод.**
- **Заёмщик и поведение необходимы всем алгоритмам**: без группы «заёмщик» AUC падает на 0,069–0,095, без «поведения» — на 0,008–0,012 (везде DeLong p < 0,001). «Поведение» здесь — то, как платит кредит, который ещё без просрочки (темп относительно графика, частичные досрочные погашения, изменения остатка), а не история просрочек.
- **География мешает двум алгоритмам и нейтральна для трёх.** Без штата, MSA и ZIP3 LightGBM намного лучше (+0,0208 [0,0166; 0,0252], p < 0,001; AUC 0,855 вместо 0,834): он переобучается на ZIP3 с большим числом категорий, на который в модели на всех 59 приходится 39 % его gain. CatBoost без неё тоже лучше (+0,0036 [0,0015; 0,0055], p < 0,001), поэтому правило удаляет группу (02_6). У логистической регрессии (−0,0003), XGBoost (−0,0003) и случайного леса (−0,0008) интервал включает 0: география им не помогает и не мешает.
- **Экспозиция, кредит и макро** для каждого алгоритма полезны или нейтральны: экспозиция от −0,0005 (логистическая регрессия, p = 0,06) до −0,0025 (XGBoost, p = 0,01), кредит от −0,0021 (LightGBM) до −0,0043 (XGBoost, p < 0,001), макро от −0,0021 (LightGBM, p = 0,007) до +0,0026 (лес, p = 0,18). **Состояние на t** (`age`) нейтрально (−0,0005…+0,0015); лишь логистическая регрессия немного выигрывает от него (−0,0002 без него, p < 0,001).

По правилу, зафиксированному заранее, отбор делается на CatBoost; удаление географии разделяет LightGBM, и ни один алгоритм без неё не становится значимо хуже, поэтому 18 признаков остаются. На этап 3 переходят проверки чувствительности: модель **с географией** и самые слабые признаки шага 8.

### 4. Selected set vs all 59 / Отобранный набор против всех 59

**EN.** The non-inferiority rule with each learner: the upper bound of the 95 % CI of ΔAUC (all 59 − selected 18) must be below 0.005.

**RU.** Правило «не хуже» на каждом алгоритме: верхняя граница 95 % интервала ΔAUC (все 59 − отобранные 18) должна быть ниже 0,005.

In [5]:
ni = []
for task in TASKS:
    for kind in KINDS:
        sel = preds[(task, kind, "candidates")] if FINAL == CAND else \
            fit_predict_validation(kind, task, FINAL, P, f"{kind}_selected", seed=SEED, train_rows=ROWS[kind])[0]
        r = compare_models({"selected": sel, "all": preds[(task, kind, "all")]}, task, "selected",
                           b=ABL_RULES["bootstrap_b"], seed=SEED).assign(learner=kind, task=task)
        ni.append(r[r["model"] == "all"])
ni = pd.concat(ni, ignore_index=True)
ni["auc_selected"] = ni["auc"] - ni["delta_auc"]
ni["non_inferior"] = ni["delta_hi"] < ABL_RULES["non_inferiority_auc"]
ni[["task", "learner", "auc_selected", "auc", "delta_auc", "delta_lo", "delta_hi", "delong_p", "non_inferior"]].rename(
    columns={"auc": "auc_all"})

,task,learner,auc_selected,auc_all,delta_auc,delta_lo,delta_hi,delong_p,non_inferior
0,y1,logistic_woe,0.8417,0.8471,0.0054,0.0026,0.0079,0.0001,False
1,y1,xgboost,0.8552,0.8561,0.0009,-0.0006,0.0025,0.0993,True
2,y1,lightgbm,0.8548,0.8326,-0.0222,-0.0262,-0.0176,0.0000,True
3,y1,random_forest,0.8392,0.8336,-0.0055,-0.0086,-0.0025,0.0373,True


**EN. Conclusion.** **The rule holds for four learners and fails only for the logistic regression.**
- Holds: CatBoost (−0.0029 [−0.0051; −0.0010], DeLong p = 0.003, from 02_6), XGBoost (+0.0009 [−0.0006; 0.0025], p = 0.10; AUC 0.855 on 18 vs 0.856 on 59), LightGBM (−0.0222 [−0.0262; −0.0176], p < 0.001; 0.855 vs 0.833) and the random forest (−0.0055 [−0.0086; −0.0025], p = 0.04; 0.839 vs 0.834). For CatBoost, LightGBM and the forest the 18 features are significantly *better* than all 59.
- Fails: the logistic regression (+0.0054 [0.0026; 0.0079], p < 0.001; AUC 0.842 on 18 vs 0.847 on 59). For this learner all 59 features are significantly better, and the upper bound is above the 0.005 margin.

This is accepted and reported as it is. The selection protocol is run on CatBoost, as fixed in advance; the other learners are a robustness check, so the 18 features stay. The loss is small (point estimate +0.005 AUC) and the next cell shows where it comes from: mainly features removed at steps 4–5 as near-duplicates of a selected feature, and the geography group — not features removed as unstable, and not the delinquency history, which is not a candidate at all.

**RU. Вывод.** **Правило выполняется для четырёх алгоритмов и не выполняется только для логистической регрессии.**
- Выполняется: CatBoost (−0,0029 [−0,0051; −0,0010], DeLong p = 0,003, из 02_6), XGBoost (+0,0009 [−0,0006; 0,0025], p = 0,10; AUC 0,855 на 18 против 0,856 на 59), LightGBM (−0,0222 [−0,0262; −0,0176], p < 0,001; 0,855 против 0,833) и случайный лес (−0,0055 [−0,0086; −0,0025], p = 0,04; 0,839 против 0,834). У CatBoost, LightGBM и леса 18 признаков значимо *лучше* всех 59.
- Не выполняется: логистическая регрессия (+0,0054 [0,0026; 0,0079], p < 0,001; AUC 0,842 на 18 против 0,847 на 59). Для неё все 59 признаков значимо лучше, и верхняя граница выше допуска 0,005.

Это принимается и показывается как есть. Протокол отбора выполняется на CatBoost, как зафиксировано заранее; другие алгоритмы — проверка устойчивости, поэтому 18 признаков остаются. Потеря небольшая (точечная оценка +0,005 AUC), и следующая ячейка показывает, откуда она: в основном из признаков, удалённых на шагах 4–5 как почти дубликаты отобранного признака, и из группы «география», — а не из признаков, удалённых как неустойчивые, и не из истории просрочек, которой среди кандидатов вообще нет.

**EN.** Which excluded features the learners lean on in the models on all 59 candidates: share of the native importance (gain for the boosting models, impurity decrease for the random forest, standardised coefficient for the logistic regression) that falls on the 41 features outside the selected set. Saved to `reports/feature_selection/other_models_importance_all.csv`.

**RU.** На какие исключённые признаки опираются модели на всех 59 кандидатах: доля собственной важности модели (gain у бустинга, уменьшение неоднородности у случайного леса, стандартизованный коэффициент у логистической регрессии), приходящаяся на 41 признак вне отобранного набора. Результат сохранён в `reports/feature_selection/other_models_importance_all.csv`.

In [6]:
# what the excluded features add: native importance share in the models on all candidates
imp = []
for (task, kind, name), (m, enc, binners, feats) in models.items():
    if name != "all":
        continue
    names = (m.get_booster().feature_names if hasattr(m, "get_booster") else
             m.booster_.feature_name() if hasattr(m, "booster_") else feats)
    v = native_importance(m)
    imp.append(pd.DataFrame({"task": task, "learner": kind, "feature": names, "share": v / v.sum()}))
imp = pd.concat(imp, ignore_index=True)
imp["selected"] = imp["feature"].isin(FINAL)
imp.to_csv(FEATURE_SELECTION_DIR / "other_models_importance_all.csv", index=False)
print(imp.groupby("learner").apply(lambda g: g.loc[~g["selected"], "share"].sum()).rename("share_of_excluded").round(3).to_string())
(imp[~imp["selected"]].sort_values("share", ascending=False).groupby("learner").head(5)
 .pivot_table(index="feature", columns="learner", values="share").round(3))


learner
lightgbm        0.5310
logistic_woe    0.4070
random_forest   0.5390
xgboost         0.1750


learner,lightgbm,logistic_woe,random_forest,xgboost
feature,,,,
current_interest_rate,0.0100,NaN,NaN,0.0230
hpi_chg_12m,NaN,NaN,0.0330,NaN
hpi_chg_since_orig,NaN,NaN,0.0350,NaN
metropolitan_statistical_area_msa_or_metropolitan_statistical_division_area_msda,0.0660,0.0220,NaN,NaN
n_curtail_12m,NaN,0.0450,NaN,0.0140
number_of_borrowers,NaN,0.0800,NaN,NaN
original_combined_loan_to_value_ratio_cltv,0.0110,NaN,NaN,0.0140
sato,0.0160,0.0370,0.0370,0.0220
upb,NaN,NaN,0.0370,0.0100


**EN. How to read and overall conclusion.** In the models on all 59 the 41 excluded features take 18 % of the native importance in XGBoost, 41 % in the logistic regression, 53 % in LightGBM and 54 % in the random forest (an even spread would give 41/59 = 69 %; the forest picks among a few random features at every split and spreads its importance over correlated twins, yet it is better on the selected 18 than on 59). Most of this share falls on two kinds of features: near-duplicates of a selected feature removed at steps 4–5 (29 of the 41 % in the logistic regression, 13 of the 18 % in XGBoost) and the geography group removed at step 9 (45 of the 53 % in LightGBM, 9 % in the logistic regression); features removed as unstable (step 6) take 2–3 % in each of these three. The largest:
- logistic regression — `number_of_borrowers` 8.0 % (|ρ| = 0.77 with the co-borrower FICO), `zip_code_short` 6.2 %, `n_curtail_12m` 4.5 % (0.90 with `n_curtail_6m`), `sato` 3.7 % (0.85 with the interest rate), MSA 2.2 %, `upb_to_orig` 2.2 % (0.78 with `upb_chg_6m`);
- XGBoost — `current_interest_rate` 2.3 % (1.00 with the original rate), `sato` 2.2 %, `n_curtail_12m` 1.4 %, CLTV 1.4 % (0.87 with `mtm_ltv`);
- LightGBM — `zip_code_short` 39 % and MSA 7 %: this is the overfitting to fine geography seen in section 3.

A correlated twin still carries a little information of its own; the redundancy step keeps one feature of each pair for a clear interpretation and gives up that little. The linear model, which cannot rebuild the twin from the kept feature, loses the most — the +0.005 AUC of section 4.

**Overall:** the main lines of the selection are not tailored to CatBoost. The borrower and behaviour groups are essential for all five learners (boosting, bagging, linear), and the selected 18 are not worse than all 59 for CatBoost, XGBoost, LightGBM and the random forest. Only for the logistic regression all 59 are slightly better (+0.005 AUC) — the cost of dropping near-duplicates (and, for this linear model, geography), reported openly; it does not come from unstable or delinquency features. Disagreements on single features are limited to weak ones (`first_time_home_buyer_indicator`, `property_type`, `seller_name`, `property_state`); geography is removed by CatBoost and LightGBM and is neutral for the other three. They become sensitivity checks in stage 3.

**RU. Как читать и общий вывод.** В моделях на всех 59 признаках на 41 исключённый приходится 18 % собственной важности у XGBoost, 41 % у логистической регрессии, 53 % у LightGBM и 54 % у случайного леса (при равномерном распределении было бы 41/59 = 69 %; лес на каждом разбиении выбирает из нескольких случайных признаков и размазывает важность по коррелированным двойникам, но при этом на отобранных 18 он лучше, чем на 59). Большая часть этой доли приходится на признаки двух видов: почти дубликаты отобранного признака, удалённые на шагах 4–5 (29 из 41 % у логистической регрессии, 13 из 18 % у XGBoost), и группу «география», удалённую на шаге 9 (45 из 53 % у LightGBM, 9 % у логистической регрессии); на признаки, удалённые как неустойчивые (шаг 6), — по 2–3 % у каждого из этих трёх. Крупнейшие:
- логистическая регрессия — `number_of_borrowers` 8,0 % (|ρ| = 0,77 с FICO созаёмщика), `zip_code_short` 6,2 %, `n_curtail_12m` 4,5 % (0,90 с `n_curtail_6m`), `sato` 3,7 % (0,85 со ставкой), MSA 2,2 %, `upb_to_orig` 2,2 % (0,78 с `upb_chg_6m`);
- XGBoost — `current_interest_rate` 2,3 % (1,00 со ставкой при выдаче), `sato` 2,2 %, `n_curtail_12m` 1,4 %, CLTV 1,4 % (0,87 с `mtm_ltv`);
- LightGBM — `zip_code_short` 39 % и MSA 7 %: это и есть переобучение на мелкой географии из раздела 3.

Коррелированный двойник всё же несёт немного собственной информации; шаг избыточности оставляет по одному признаку из пары ради ясной интерпретации и жертвует этой малостью. Больше всех теряет линейная модель, которая не может восстановить двойника из оставленного признака, — это и есть +0,005 AUC из раздела 4.

**Итог:** основные решения отбора не подстроены под CatBoost. Группы «заёмщик» и «поведение» необходимы всем пяти алгоритмам (бустинг, бэггинг, линейная модель), а отобранные 18 признаков не хуже всех 59 для CatBoost, XGBoost, LightGBM и случайного леса. Только для логистической регрессии все 59 чуть лучше (+0,005 AUC) — цена удаления почти дубликатов (а для этой линейной модели и географии), она показана открыто; её дают не неустойчивые признаки и не признаки просрочек. Расхождения по отдельным признакам касаются только слабых (`first_time_home_buyer_indicator`, `property_type`, `seller_name`, `property_state`); географию удаляют CatBoost и LightGBM, для остальных трёх она нейтральна. На этапе 3 они переходят в проверки чувствительности.

In [7]:
aucs.to_csv(FEATURE_SELECTION_DIR / "other_models_auc.csv", index=False)
step8.to_csv(FEATURE_SELECTION_DIR / "other_models_step8.csv", index=False)
both9.to_csv(FEATURE_SELECTION_DIR / "other_models_step9.csv", index=False)
ni.to_csv(FEATURE_SELECTION_DIR / "other_models_non_inferiority.csv", index=False)
print("saved other_models_*.csv")

saved other_models_*.csv
